In [3]:
import os
import piexif
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy.stats import entropy
import cv2

In [4]:
valid_exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff")

def extract_exif_info(image_path, img_format):
   if img_format not in ["JPEG", "TIFF"]:
       return {"Make": None, "Model": None, "Software": None, "DateTimeOriginal": None}
   try:
       exif_dict = piexif.load(image_path)
       exif_data = {}
       for ifd in exif_dict:
           for tag in exif_dict[ifd]:
               key = piexif.TAGS[ifd][tag]["name"]
               value = exif_dict[ifd][tag]
               if isinstance(value, bytes):
                   value = value.decode(errors="ignore")
               exif_data[key] = value
       return {
           "Make": exif_data.get("Make", None),
           "Model": exif_data.get("Model", None),
           "Software": exif_data.get("Software", None),
           "DateTimeOriginal": exif_data.get("DateTimeOriginal", None)
       }
   except:
       return {"Make": None, "Model": None, "Software": None, "DateTimeOriginal": None}

def extract_qtable(image_path, img_format):
   if img_format != "JPEG":
       return {"qtable_mean": None, "qtable_std": None, "qtable_max": None, "qtable_min": None}
   try:
       img = Image.open(image_path)
       qt = img.quantization
       if not qt:
           return {"qtable_mean": None, "qtable_std": None}
       q = list(qt.values())[0]
       q = np.array(q)
       return {
           "qtable_mean": np.mean(q),
           "qtable_std": np.std(q),
           "qtable_max": np.max(q),
           "qtable_min": np.min(q)
       }
   except:
       return {"qtable_mean": None, "qtable_std": None, "qtable_max": None, "qtable_min": None}

def extract_entropy(img_array):
   r, g, b = img_array[:, :, 0], img_array[:, :, 1], img_array[:, :, 2]
   return {
       "r_entropy": entropy(np.histogram(r, bins=256)[0] + 1e-7),
       "g_entropy": entropy(np.histogram(g, bins=256)[0] + 1e-7),
       "b_entropy": entropy(np.histogram(b, bins=256)[0] + 1e-7)
   }

def extract_blur_sharpness(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
   return {"blur_metric": lap_var}

def extract_hu_moments(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   moments = cv2.moments(gray)
   hu_moments = cv2.HuMoments(moments).flatten()
   return {f"hu_{i+1}": float(val) for i, val in enumerate(hu_moments)}

def extract_dct_features(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   dct = cv2.dct(np.float32(gray) / 255.0)
   dct_abs = np.abs(dct)
   return {
       "dct_mean": np.mean(dct_abs),
       "dct_std": np.std(dct_abs),
       "dct_max": np.max(dct_abs),
       "dct_energy": np.sum(dct_abs ** 2)
   }

def extract_noise_features(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   blurred = cv2.GaussianBlur(gray, (3, 3), 0)
   noise = gray.astype(np.float32) - blurred.astype(np.float32)
   return {
       "noise_mean": np.mean(noise),
       "noise_std": np.std(noise)
   }

def extract_edge_density(img_array):
   gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
   edges = cv2.Canny(gray, 100, 200)
   return {
       "edge_density": np.sum(edges > 0) / edges.size
   }

def extract_image_stats(img):
   return {
       "width": img.width,
       "height": img.height
   }

def process_images(folder, label):
   data = []
   for fname in tqdm(os.listdir(folder), desc=f"Processing {folder}"):
       if not fname.lower().endswith(valid_exts):
           continue
       full_path = os.path.join(folder, fname)
       try:
           img = Image.open(full_path).convert("RGB")
           img_array = np.array(img)
           img_format = img.format or os.path.splitext(fname)[-1].replace(".", "").upper()

           row = {
               "filename": fname,
               "label": label,
               "format": img_format,
               "has_exif": img_format in ['JPEG', 'TIFF'],
               "has_qtable": img_format == 'JPEG'
           }

           row.update(extract_exif_info(full_path, img_format))
           row.update(extract_image_stats(img))
           row.update(extract_entropy(img_array))
           row.update(extract_blur_sharpness(img_array))
           row.update(extract_hu_moments(img_array))
           row.update(extract_dct_features(img_array))
           row.update(extract_noise_features(img_array))
           row.update(extract_edge_density(img_array))
           row.update(extract_qtable(full_path, img_format))

           data.append(row)
       except Exception as e:
           print(f"[ERROR] {fname}: {e}")
   return data